# Fine-Tuning Programming Agent with Before/After Testing

This notebook demonstrates:
1. **Loading** the R2E-Gym agent dataset
2. **Testing BEFORE** training
3. **Fine-tuning** the model
4. **Testing AFTER** training
5. **Comparing** the results

## 📦 Installation (Run Once)

In [1]:
%%capture
# Install required packages
!pip install unsloth
!pip install transformers==4.56.2
!pip install trl==0.22.2
!pip install datasets accelerate bitsandbytes

## 🔧 Configuration

In [2]:
# Configuration
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"
DATASET_NAME = "R2E-Gym/R2EGym-TestingAgent-SFT-Trajectories"
MAX_SEQ_LENGTH = 4096
OUTPUT_DIR = "./finetuned_agent"

# Training settings
BATCH_SIZE = 2
GRAD_ACCUM = 4
MAX_STEPS = 50  # Increase for better results
LEARNING_RATE = 2e-4

print("✅ Configuration loaded")

✅ Configuration loaded


## 📊 Load Dataset

In [3]:
import json
from datasets import load_dataset

print("Loading dataset...")
dataset = load_dataset(DATASET_NAME, split="train")
print(f"✅ Loaded {len(dataset)} examples")

# Show first example
print("\n📋 First Example:")
first = dataset[0]
print(f"Exit reason: {first['exit_reasons']}")
print(f"Num steps: {first['num_steps']}")

messages = json.loads(first['messages']) if isinstance(first['messages'], str) else first['messages']
print(f"\nFirst message: {messages[0]['content'][:200]}...")

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/428 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/26.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2281 [00:00<?, ? examples/s]

✅ Loaded 2281 examples

📋 First Example:
Exit reason: agent
Num steps: 3

First message: You are a programming agent who is provided a github issue and repository bash environment and is tasked to generate a standalone test script that can reproduce and verify the issue without relying on...


## 🤖 Load Model

In [4]:
import torch
from unsloth import FastLanguageModel

print("Loading model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
print("✅ Model loaded")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model...
==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

✅ Model loaded


## 🔍 Test BEFORE Training

In [5]:
# Get test prompt from first example
messages = json.loads(first['messages']) if isinstance(first['messages'], str) else first['messages']
test_prompt = next((msg['content'] for msg in messages if msg['role'] == 'user'), 
                   "Write a Python function to calculate factorial.")

print("="*80)
print("TESTING BEFORE TRAINING")
print("="*80)
print(f"\n📝 Test Prompt:\n{test_prompt[:300]}...")

# Generate response
FastLanguageModel.for_inference(model)
test_messages = [{"role": "user", "content": test_prompt}]
inputs = tokenizer.apply_chat_template(
    test_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs_before = model.generate(input_ids=inputs, max_new_tokens=300, temperature=0.7)
response_before = tokenizer.decode(outputs_before[0], skip_special_tokens=True)

print(f"\n💬 Response BEFORE Training:")
print("-" * 80)
print(response_before[:500])
print("..." if len(response_before) > 500 else "")
print("-" * 80)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


TESTING BEFORE TRAINING

📝 Test Prompt:
Consider the following github issue:
<github_issue>

**Title:** Categorical.from_codes Incorrectly Handles Nullable Int Arrays and NA Values

**Description:**
When using `Categorical.from_codes` with nullable integer arrays (`Int64` dtype), the method raises unexpected errors, even when the input do...

💬 Response BEFORE Training:
--------------------------------------------------------------------------------
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Consider the following github issue:
<github_issue>

**Title:** Categorical.from_codes Incorrectly Handles Nullable Int Arrays and NA Values

**Description:**
When using `Categorical.from_codes` with nullable integer arrays (`Int64` dtype), the method raises unexpected errors, even when the input does not contain `NA` values. Additionally, when `NA` values are present, the error message does not accurately reflect t
...
-----------------------------------------

## ⚙️ Prepare for Training

In [6]:
print("Preparing model for training...")

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", 
                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("✅ LoRA adapters added")

Preparing model for training...


Unsloth 2026.2.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


✅ LoRA adapters added


## 📝 Format Dataset

In [7]:
def format_data(examples):
    texts = []
    for msg_str in examples["messages"]:
        messages = json.loads(msg_str) if isinstance(msg_str, str) else msg_str
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

print("Formatting dataset...")
dataset = dataset.map(format_data, batched=True, num_proc=2)
print(f"✅ Dataset formatted: {len(dataset)} examples")

Formatting dataset...


Map (num_proc=2):   0%|          | 0/2281 [00:00<?, ? examples/s]

✅ Dataset formatted: 2281 examples


## 🎯 Train Model

In [8]:
from transformers import TrainingArguments
from trl import SFTTrainer

print("Setting up trainer...")
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        max_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir=OUTPUT_DIR,
        report_to="none",
    ),
)

print("\n" + "="*80)
print("🎯 STARTING TRAINING")
print("="*80)

stats = trainer.train()

print("\n" + "="*80)
print("✅ TRAINING COMPLETE")
print("="*80)
print(f"Final loss: {stats.training_loss:.4f}")

Setting up trainer...


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2281 [00:00<?, ? examples/s]


🎯 STARTING TRAINING


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,281 | Num Epochs = 1 | Total steps = 50
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss
5,2.165800
10,1.666600
15,1.246400
20,0.742100
25,0.249000
30,0.034700
35,0.006300
40,0.002900
45,0.001800
50,0.001500



✅ TRAINING COMPLETE
Final loss: 0.6117


## 🔍 Test AFTER Training

In [9]:
print("="*80)
print("TESTING AFTER TRAINING")
print("="*80)

FastLanguageModel.for_inference(model)

# Same prompt as before
inputs = tokenizer.apply_chat_template(
    test_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

print(f"\n📝 Test Prompt (same as before):\n{test_prompt[:300]}...")

outputs_after = model.generate(input_ids=inputs, max_new_tokens=300, temperature=0.7)
response_after = tokenizer.decode(outputs_after[0], skip_special_tokens=True)

print(f"\n💬 Response AFTER Training:")
print("-" * 80)
print(response_after[:500])
print("..." if len(response_after) > 500 else "")
print("-" * 80)

TESTING AFTER TRAINING

📝 Test Prompt (same as before):
Consider the following github issue:
<github_issue>

**Title:** Categorical.from_codes Incorrectly Handles Nullable Int Arrays and NA Values

**Description:**
When using `Categorical.from_codes` with nullable integer arrays (`Int64` dtype), the method raises unexpected errors, even when the input do...

💬 Response AFTER Training:
--------------------------------------------------------------------------------
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Consider the following github issue:
<github_issue>

**Title:** Categorical.from_codes Incorrectly Handles Nullable Int Arrays and NA Values

**Description:**
When using `Categorical.from_codes` with nullable integer arrays (`Int64` dtype), the method raises unexpected errors, even when the input does not contain `NA` values. Additionally, when `NA` values are present, the error message does not accurately reflect t
...
--------------------------

## 📊 Compare Results

In [11]:
print("="*80)
print("📊 BEFORE vs AFTER COMPARISON")
print("="*80)

print("\n🔴 BEFORE Training:")
print("-" * 80)
print(response_before[:])
print("...")

print("\n🟢 AFTER Training:")
print("-" * 80)
print(response_after[:])
print("...")

📊 BEFORE vs AFTER COMPARISON

🔴 BEFORE Training:
--------------------------------------------------------------------------------
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Consider the following github issue:
<github_issue>

**Title:** Categorical.from_codes Incorrectly Handles Nullable Int Arrays and NA Values

**Description:**
When using `Categorical.from_codes` with nullable integer arrays (`Int64` dtype), the method raises unexpected errors, even when the input does not contain `NA` values. Additionally, when `NA` values are present, the error message does not accurately reflect the issue.

**Example Code:**
```python
import pandas as pd
from pandas.core.arrays.categorical import Categorical

# Example with nullable Int array without NA values
codes = pd.array([0, 1], dtype='Int64')
categories = ['a', 'b']
categorical = Categorical.from_codes(codes, categories=categories)
```

**Expected Behavior:**
- The `Categorical.from_codes` method should

## 💾 Save Model

In [12]:
print("Saving model...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved to: {OUTPUT_DIR}")

Saving model...
✅ Model saved to: ./finetuned_agent


## 🎁 Test with Additional Examples

In [ ]:
print("="*80)
print("🎁 TESTING WITH ADDITIONAL EXAMPLES")
print("="*80)

test_prompts = [
    "Write a Python function to check if a string is a palindrome.",
    "Create a bash script to count files in a directory.",
    "How do I debug a segmentation fault in C?",
]

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n[Test {i}] {prompt}")
    
    msgs = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(inputs, max_new_tokens=200, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f"\nResponse:")
    print("-" * 80)
    print(response[:400])
    print("..." if len(response) > 400 else "")

## 🎉 Summary

In [ ]:
print("="*80)
print("🎉 FINE-TUNING COMPLETE!")
print("="*80)
print(f"✓ Trained on {len(dataset)} examples")
print(f"✓ Model saved to: {OUTPUT_DIR}")
print(f"✓ Final training loss: {stats.training_loss:.4f}")
print("\n✨ Your model is ready to use for programming agent tasks!")